<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_1_Neural_Networks/19_1_9_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks: Exercise
## Wine Quality Prediction

---

## Overview

In exercise 18_5_9, you built an XGBoost classifier to predict wine quality — three classes (Low, Medium, High) from 11 chemical measurements. In this exercise, you will apply the full module 19 pipeline to the same dataset: build a PyTorch neural network, train it, and compare your results to what you found before.

The dataset is familiar. The task is identical. The only change is the model.

**Your tasks:**
1. Load the data and recreate the three-class target
2. Convert to tensors and build a `DataLoader`
3. Define a `WineNet` as an `nn.Module`
4. Train with `CrossEntropyLoss`, balanced class weights, and Adam
5. Plot training and validation loss curves — does your model overfit?
6. Compute the classification report and report macro F1
7. Compare to your XGBoost result from 18_5_9
8. Answer the reflection questions

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import compute_class_weight

import seaborn as sns
sns.set_theme(style='whitegrid')
torch.manual_seed(42)

class_names = ['Low', 'Medium', 'High']

---

## Task 1: Load and Prepare the Data

Load the red wine quality dataset and create a three-class target:
- **Low**: quality ≤ 5
- **Medium**: quality = 6
- **High**: quality ≥ 7

The separator is a semicolon (`;`). This is the same binning you did in exercise 18_5_9.

In [2]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

# TODO: Load the data
# wine = pd.read_csv(url, sep=';')

# TODO: Create the three-class target column
# def assign_group(q):
#     if q <= 5:  return 'Low'
#     elif q == 6: return 'Medium'
#     else:        return 'High'
# wine['quality_group'] = wine['quality'].apply(assign_group)

# TODO: Print the class distribution
# print(wine['quality_group'].value_counts())

In [3]:
# @title Execute to see solution
solution = '''
wine = pd.read_csv(url, sep=";")

def assign_group(q):
    if q <= 5:   return "Low"
    elif q == 6: return "Medium"
    else:        return "High"

wine["quality_group"] = wine["quality"].apply(assign_group)
print(f"Dataset: {wine.shape[0]} wines, {wine.shape[1]-2} features")
print(wine["quality_group"].value_counts())

naive_baseline = wine["quality_group"].value_counts().max() / len(wine)
print(f"Naive baseline (always predict most common class): {naive_baseline:.1%}")
'''
print(solution)


wine = pd.read_csv(url, sep=";")

def assign_group(q):
    if q <= 5:   return "Low"
    elif q == 6: return "Medium"
    else:        return "High"

wine["quality_group"] = wine["quality"].apply(assign_group)
print(f"Dataset: {wine.shape[0]} wines, {wine.shape[1]-2} features")
print(wine["quality_group"].value_counts())

naive_baseline = wine["quality_group"].value_counts().max() / len(wine)
print(f"Naive baseline (always predict most common class): {naive_baseline:.1%}")



---

## Task 2: Convert to Tensors and Build a DataLoader

1. Create integer labels: Low=0, Medium=1, High=2
2. Extract features (all columns except `quality` and `quality_group`) as `X`
3. Do a 60/20/20 train/val/test split with stratification
4. Apply `StandardScaler` (fit on train only)
5. Convert to PyTorch tensors — remember that `CrossEntropyLoss` needs `y` as `torch.long`
6. Create a `DataLoader` for the training set with `batch_size=32` and `shuffle=True`

In [4]:
# TODO: Create X (numpy) and y (numpy, integer encoded)
# label_map = {name: i for i, name in enumerate(class_names)}
# y_raw = wine['quality_group'].map(label_map).to_numpy(dtype=np.int64)
# X_raw = wine.drop(columns=['quality', 'quality_group']).to_numpy(dtype=np.float32)

# TODO: Three-way split
# X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42)
# X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

# TODO: Standardize
# scaler = StandardScaler()
# ...

# TODO: Convert to tensors
# X_train, y_train, etc.

# TODO: Build DataLoader
# BATCH_SIZE = 32
# train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

In [5]:
# @title Execute to see solution
solution = '''
label_map = {name: i for i, name in enumerate(class_names)}
y_raw = wine["quality_group"].map(label_map).to_numpy(dtype=np.int64)
X_raw = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=np.float32)

X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42
)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42
)

scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np)
X_val   = torch.tensor(X_val_np)
X_test  = torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np)
y_val   = torch.tensor(y_val_np)
y_test  = torch.tensor(y_test_np)

BATCH_SIZE   = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"y dtype: {y_train.dtype}")
'''
print(solution)


label_map = {name: i for i, name in enumerate(class_names)}
y_raw = wine["quality_group"].map(label_map).to_numpy(dtype=np.int64)
X_raw = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=np.float32)

X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42
)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42
)

scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np)
X_val   = torch.tensor(X_val_np)
X_test  = torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np)
y_val   = torch.tensor(y_val_np)
y_test  = torch.tensor(y_test_np)

BATCH_SIZE   = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape

---

## Task 3: Define WineNet

Define a `WineNet(nn.Module)` class with at least two hidden layers. You choose the hidden layer sizes.

Remember: for `CrossEntropyLoss`, the output layer should be `nn.Linear(last_hidden, 3)` — **no softmax** at the end.

In [6]:
# TODO: Define WineNet
# class WineNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         ...

# TODO: Instantiate the model and print the total parameter count
# model = WineNet()
# total = sum(p.numel() for p in model.parameters())
# print(f'Total parameters: {total:,}')

In [7]:
# @title Execute to see solution
solution = '''
class WineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)   # 3 outputs — NO softmax
        )

    def forward(self, x):
        return self.net(x)

model = WineNet()
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
print(model)
'''
print(solution)


class WineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)   # 3 outputs — NO softmax
        )

    def forward(self, x):
        return self.net(x)

model = WineNet()
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
print(model)



---

## Task 4: Compute Class Weights and Train

1. Use `compute_class_weight('balanced', ...)` on `y_train_np` to get balanced weights
2. Train for 200 epochs with `CrossEntropyLoss(weight=class_weights)` and Adam
3. Track and store training loss and validation macro F1 each epoch

In [8]:
# TODO: Compute class weights
# class_weights_np = compute_class_weight('balanced', classes=np.arange(3), y=y_train_np)
# class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

# TODO: Set up model, optimizer, loss
# torch.manual_seed(42)
# model = WineNet()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
# criterion = nn.CrossEntropyLoss(weight=class_weights)

# TODO: Training loop (200 epochs)
# train_losses = []
# val_f1s      = []
# for epoch in range(1, 201):
#     model.train()
#     ...
#     model.eval()
#     with torch.no_grad():
#         val_preds = model(X_val).argmax(dim=1).numpy()
#     val_f1s.append(f1_score(y_val_np, val_preds, average='macro'))

In [9]:
# @title Execute to see solution
solution = '''
class_weights_np = compute_class_weight("balanced", classes=np.arange(3), y=y_train_np)
class_weights    = torch.tensor(class_weights_np, dtype=torch.float32)

torch.manual_seed(42)
model     = WineNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_losses, val_f1s = [], []

for epoch in range(1, 201):
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val).argmax(dim=1).numpy()
    val_f1s.append(f1_score(y_val_np, val_preds, average="macro"))
    model.train()

print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val macro F1: {val_f1s[-1]:.3f}")
'''
print(solution)


class_weights_np = compute_class_weight("balanced", classes=np.arange(3), y=y_train_np)
class_weights    = torch.tensor(class_weights_np, dtype=torch.float32)

torch.manual_seed(42)
model     = WineNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_losses, val_f1s = [], []

for epoch in range(1, 201):
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val).argmax(dim=1).numpy()
    val_f1s.append(f1_score(y_val_np, val_preds, average="macro"))
    model.train()

print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val macro F1: {val_f1s[-1]:.3f}")



---

## Task 5: Plot the Loss and F1 Curves

Plot training loss and validation macro F1 on separate axes (or separate figures). Look at the curves and answer: does your model appear to be overfitting? How can you tell from the curves?

In [10]:
# TODO: Plot training loss and validation macro F1
# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# axes[0].plot(train_losses, ...)
# axes[1].plot(val_f1s, ...)

In [11]:
# @title Execute to see solution
solution = '''
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(train_losses, color="steelblue", lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Training Loss")

axes[1].plot(val_f1s, color="#C0392B", lw=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Validation Macro F1")
axes[1].set_ylim(0, 1)

plt.suptitle("WineNet Training Curves", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
'''
print(solution)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(train_losses, color="steelblue", lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Training Loss")

axes[1].plot(val_f1s, color="#C0392B", lw=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Validation Macro F1")
axes[1].set_ylim(0, 1)

plt.suptitle("WineNet Training Curves", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()



---

## Task 6: Evaluate on the Test Set

Print the full `classification_report` and report the test macro F1.

In [12]:
# TODO: Evaluate on X_test
# model.eval()
# with torch.no_grad():
#     test_preds = model(X_test).argmax(dim=1).numpy()
# print(classification_report(y_test_np, test_preds, target_names=class_names, digits=3))

In [13]:
# @title Execute to see solution
solution = '''
model.eval()
with torch.no_grad():
    test_preds = model(X_test).argmax(dim=1).numpy()

print(classification_report(y_test_np, test_preds, target_names=class_names, digits=3))
print(f"Test macro F1: {f1_score(y_test_np, test_preds, average='macro'):.3f}")
'''
print(solution)


model.eval()
with torch.no_grad():
    test_preds = model(X_test).argmax(dim=1).numpy()

print(classification_report(y_test_np, test_preds, target_names=class_names, digits=3))
print(f"Test macro F1: {f1_score(y_test_np, test_preds, average='macro'):.3f}")



---

## Task 7: Stratified Cross-Validation

A single train/val/test split can be misleading, especially when minority classes are small. Run 5-fold stratified cross-validation on the **balanced** WineNet and report the mean and standard deviation of macro F1 across folds.

Compare this CV estimate to your single-split result from Task 6.

In [14]:
# TODO: 5-fold stratified CV
# outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv_scores = []
# for train_idx, test_idx in outer_cv.split(X_raw, y_raw):
#     ...  # standardize, to tensor, train WineNet, evaluate macro F1
# print(f'CV macro F1: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}')

In [15]:
# @title Execute to see solution
solution = '''
outer_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_raw, y_raw)):
    X_tr_np, X_te_np = X_raw[train_idx], X_raw[test_idx]
    y_tr_np, y_te_np = y_raw[train_idx], y_raw[test_idx]

    sc = StandardScaler()
    X_tr_np = sc.fit_transform(X_tr_np)
    X_te_np = sc.transform(X_te_np)

    X_tr = torch.tensor(X_tr_np)
    X_te = torch.tensor(X_te_np)
    y_tr = torch.tensor(y_tr_np)

    cw_np = compute_class_weight("balanced", classes=np.arange(3), y=y_tr_np)
    cw    = torch.tensor(cw_np, dtype=torch.float32)

    torch.manual_seed(42)
    m   = WineNet()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=cw)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    for _ in range(200):
        m.train()
        for Xb, yb in loader:
            opt.zero_grad()
            crit(m(Xb), yb).backward()
            opt.step()

    m.eval()
    with torch.no_grad():
        preds = m(X_te).argmax(dim=1).numpy()
    cv_scores.append(f1_score(y_te_np, preds, average="macro"))
    print(f"  Fold {fold+1}: macro F1 = {cv_scores[-1]:.3f}")

cv_scores = np.array(cv_scores)
print(f"\nCV macro F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
'''
print(solution)


outer_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_raw, y_raw)):
    X_tr_np, X_te_np = X_raw[train_idx], X_raw[test_idx]
    y_tr_np, y_te_np = y_raw[train_idx], y_raw[test_idx]

    sc = StandardScaler()
    X_tr_np = sc.fit_transform(X_tr_np)
    X_te_np = sc.transform(X_te_np)

    X_tr = torch.tensor(X_tr_np)
    X_te = torch.tensor(X_te_np)
    y_tr = torch.tensor(y_tr_np)

    cw_np = compute_class_weight("balanced", classes=np.arange(3), y=y_tr_np)
    cw    = torch.tensor(cw_np, dtype=torch.float32)

    torch.manual_seed(42)
    m   = WineNet()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=cw)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    for _ in range(200):
        m.train()
        for Xb, yb in loader:
            opt.zero_grad()
            crit(m(Xb), yb).backward()
      

---

## Task 8: Reflection Questions

Answer these questions in the markdown cells below. Double-click to edit.

**Q1.** Compare your WineNet macro F1 to the XGBoost result from exercise 18_5_9. Which model performed better? What might explain the difference?

**Q2.** You saw training loss and validation macro F1 curves during training. What advantages does watching these curves give you that you did not have when using XGBoost's `.fit()`?

**Q3.** The wine dataset has about 1,600 samples. Based on what you observed (loss curves, CV variance, etc.), does a neural network seem like the right choice for a dataset this size? What would need to change for a neural network to clearly outperform XGBoost here?

**Q4.** Describe one concrete change that would likely *increase* your model's capacity (make it more expressive) and one change that would likely *decrease* overfitting. These could be the same change or different changes.

**Q5.** In notebook 19_0_3, you saw gradient descent find essentially the same regression line as sklearn's analytic solution. In this exercise, there is no analytic solution — the network can only be trained iteratively. Does that mean the neural network is a strictly worse tool for small tabular datasets? When would you choose it anyway?

*Write your answer to Q1 here.*

*Write your answer to Q2 here.*

*Write your answer to Q3 here.*

*Write your answer to Q4 here.*

*Write your answer to Q5 here.*

---

## Putting It All Together

You applied the complete module 19 pipeline end-to-end:

| Step | What you did |
|---|---|
| Data prep | Three-way split, StandardScaler, `torch.long` labels |
| Model | `WineNet(nn.Module)` with two hidden layers |
| Loss | `CrossEntropyLoss(weight=class_weights)` for imbalance |
| Optimizer | Adam with `weight_decay` |
| Training | Mini-batch loop with `DataLoader`, 200 epochs |
| Monitoring | Training loss and validation macro F1 curves |
| Evaluation | `classification_report` on held-out test set |
| Validation | Stratified K-Fold CV for honest estimate |

This is the same workflow you would use on any tabular multiclass dataset — change the architecture and the number of features, and everything else stays the same.